In [8]:
import os
from time import sleep
from tqdm import tqdm

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import StandardScaler

from keras.models import Model, load_model
from keras.layers import Input, Dense, LSTM, RepeatVector, TimeDistributed
from keras import regularizers

from matplotlib import pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

In [9]:
base_path = '/Users/nicholastey/Desktop/thesis/gnss_spoof_detector/spoof_detector/data/'
base_path = 'C:/Users/nicho/Desktop/gnss_spoof_detector/spoof_detector/data'

In [10]:
cs_path, ds1_path, ds3_path, ds2_path, ds4_path, ds7_path, ds8_path = [os.path.join(base_path, ds_path) for ds_path in os.listdir(base_path)]


In [11]:
columns = ['channel', 'prn', 'acq_doopler_hz', 'acq_doppler_step', 'fs', 'prompt_i', 'prompt_q', 'cn0_db_hz', 'carrier_doppler_hz', 'pseudorange_m', 'rx_time']

for ds_fname in os.listdir(base_path):
  tmp_path = os.path.join(base_path, ds_fname)
  key = ds_fname.split('.')[0]
  
  tmp_df = pd.read_csv(tmp_path, header=None)
  prn_li = []
  for i in range(8):
    prn_li.append(int(tmp_df.iloc[1, 11*i+1]))
  globals()[f'{key}_dict'] = {
      int(f'{prn_li[0]}') : tmp_df.iloc[100:, 0:9].rename(columns=lambda x: columns[x%11]).iloc[:, 5:].diff().fillna(0),
      int(f'{prn_li[1]}') : tmp_df.iloc[100:, 11:20].rename(columns=lambda x: columns[x%11]).iloc[:, 5:].diff().fillna(0),
      int(f'{prn_li[2]}') : tmp_df.iloc[100:, 22:31].rename(columns=lambda x: columns[x%11]).iloc[:, 5:].diff().fillna(0),
      int(f'{prn_li[3]}') : tmp_df.iloc[100:, 33:42].rename(columns=lambda x: columns[x%11]).iloc[:, 5:].diff().fillna(0),
      int(f'{prn_li[4]}') : tmp_df.iloc[100:, 44:53].rename(columns=lambda x: columns[x%11]).iloc[:, 5:].diff().fillna(0),
      int(f'{prn_li[5]}') : tmp_df.iloc[100:, 55:64].rename(columns=lambda x: columns[x%11]).iloc[:, 5:].diff().fillna(0),
      int(f'{prn_li[6]}') : tmp_df.iloc[100:, 66:75].rename(columns=lambda x: columns[x%11]).iloc[:, 5:].diff().fillna(0),
      int(f'{prn_li[7]}') : tmp_df.iloc[100:, 77:86].rename(columns=lambda x: columns[x%11]).iloc[:, 5:].diff().fillna(0),
      'prn' : prn_li
  }

dicts = [cs_dict, ds1_dict, ds2_dict, ds3_dict, ds4_dict, ds7_dict, ds8_dict]

In [12]:
big_df = []

dicts_with_labels = [
    (cs_dict, 0),  # Clean
    (ds1_dict, 1),
    (ds2_dict, 2),
    (ds3_dict, 3),
    (ds4_dict, 4),
    (ds7_dict, 5),
    (ds8_dict, 6)
]

for dataset, label in dicts_with_labels:
    for prn in dataset['prn']:
        df = dataset[prn].copy()
        df['spoofed'] = label
        df['prn'] = prn
        big_df.append(df)

big_df = pd.concat(big_df, ignore_index=True)
print(big_df.shape)
print(big_df.columns)
print(big_df['spoofed'].value_counts(normalize=True))

big_df

(189272, 6)
Index(['prompt_i', 'prompt_q', 'cn0_db_hz', 'carrier_doppler_hz', 'spoofed',
       'prn'],
      dtype='object')
spoofed
5    0.146963
6    0.146963
1    0.145272
4    0.143920
2    0.142060
3    0.138002
0    0.136819
Name: proportion, dtype: float64


,prompt_i,prompt_q,cn0_db_hz,carrier_doppler_hz,spoofed,prn
0,0.0,0.0000,0.000000,0.00000,0,13
1,0.0,0.0000,0.000000,0.00000,0,13
2,0.0,0.0000,0.000000,0.00000,0,13
3,0.0,0.0000,0.000000,0.00000,0,13
4,0.0,0.0000,0.000000,0.00000,0,13
...,...,...,...,...,...,...
189267,0.0,0.0000,0.000000,0.00000,6,16
189268,0.0,0.0000,0.000000,0.00000,6,16
189269,0.0,0.0000,0.000000,0.00000,6,16
189270,0.0,0.0000,0.000000,0.00000,6,16


In [13]:
big_df.drop(columns=['prn'])

,prompt_i,prompt_q,cn0_db_hz,carrier_doppler_hz,spoofed
0,0.0,0.0000,0.000000,0.00000,0
1,0.0,0.0000,0.000000,0.00000,0
2,0.0,0.0000,0.000000,0.00000,0
3,0.0,0.0000,0.000000,0.00000,0
4,0.0,0.0000,0.000000,0.00000,0
...,...,...,...,...,...
189267,0.0,0.0000,0.000000,0.00000,6
189268,0.0,0.0000,0.000000,0.00000,6
189269,0.0,0.0000,0.000000,0.00000,6
189270,0.0,0.0000,0.000000,0.00000,6


In [14]:
X=big_df.drop(columns=['spoofed'])
y=big_df['spoofed']

In [15]:
import pywt

def apply_swt_per_sequence(X_seq, wavelet='db4', level=1, include_approx=True):
    """
    Apply Stationary Wavelet Transform (SWT) to each sequence.
    Keeps time alignment so output is suitable for LSTM.

    Input:
        X_seq: shape (n_windows, time_steps, n_features)
    Output:
        X_wavelet: shape (n_windows, time_steps, n_features_transformed)
    """
    X_wavelet = []

    for sequence in X_seq:  # (time_steps, n_features)
        seq_features = []
        for i in range(sequence.shape[1]):  # for each feature
            coeffs = pywt.swt(sequence[:, i], wavelet=wavelet, level=level, start_level=0)
            # coeffs = list of (cA, cD), each length = time_steps
            detail_list = [cD for (cA, cD) in coeffs]
            if include_approx:
                detail_list.append(coeffs[-1][0])  # final approximation
            feature_matrix = np.stack(detail_list, axis=1)  # (time_steps, n_coeffs_per_feature)
            seq_features.append(feature_matrix)
        # concat all features along last axis
        seq_features = np.concatenate(seq_features, axis=1)  # (time_steps, n_features * n_coeffs_per_feature)
        X_wavelet.append(seq_features)

    return np.array(X_wavelet)



In [16]:
def create_sequences(X, y, time_steps=10):
    Xs, ys = [], []
    for i in range(len(X) - time_steps):
        Xs.append(X.iloc[i:i + time_steps].values)
        ys.append(y.iloc[i + time_steps])  # label is at end of window
    return np.array(Xs), np.array(ys)

In [ ]:
from sklearn.model_selection import train_test_split


time_steps = 50
X_seq, y_seq = create_sequences(X, y, time_steps=time_steps)

X_seq_wavelet = apply_swt_per_sequence(X_seq)

X_train, X_test, y_train, y_test = train_test_split(X_seq_wavelet, y_seq, test_size=0.2, random_state=42, stratify=y_seq)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42, stratify=y_train)


In [ ]:
from tensorflow.keras.layers import LSTM, Dropout, Dense, BatchNormalization, Bidirectional, Input
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ReduceLROnPlateau

def create_lstm_model(input_shape):
    inputs = Input(shape=input_shape)
    x = LSTM(64, return_sequences=True)(inputs)
    x = Dropout(0.3)(x)  # Increased dropout
    x = LSTM(25, return_sequences=False)(x)
    x = Dropout(0.3)(x)  # Increased dropout
    x = Dense(25, activation='relu')(x)
    x = Dropout(0.3)(x)  # Increased dropout
    x = Dense(12, activation='relu')(x)
    x = BatchNormalization()(x)  # Batch Normalization
    outputs = Dense(7, activation='softmax')(x)

    model = Model(inputs, outputs)
    optimizer = Adam(learning_rate=0.01)  # Lower learning rate
    model.compile(optimizer=optimizer, loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    
    return model

lr_schedule = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)


In [ ]:
lstm_model = create_lstm_model((X_train.shape[1], X_train.shape[2]))

In [ ]:
# Train LSTM model
lstm_model.fit(X_train, y_train, 
               validation_data=(X_val, y_val), 
               epochs=1, 
               batch_size=500, 
               verbose=1, 
               callbacks=[lr_schedule])

In [ ]:
# Evaluate LSTM model
lstm_loss, lstm_accuracy = lstm_model.evaluate(X_test, y_test, verbose=1)
print(f"LSTM Test Loss: {lstm_loss:.4f}, Test Accuracy: {lstm_accuracy:.4f}")
y_pred_lstm = np.argmax(lstm_model.predict(X_test), axis=1)
print(classification_report(y_test, y_pred_lstm))
cm_lstm = confusion_matrix(y_test, y_pred_lstm)
plt.figure(figsize=(10, 8))
sns.heatmap(cm_lstm, annot=True, fmt='d', cmap='Blues')
plt.title('LSTM Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')


In [ ]:
from tensorflow.keras import layers, models, Input

def tcn_block(x, filters, kernel_size=3, dilation_rate=1, dropout=0.2):
    # First dilated conv
    conv1 = layers.Conv1D(filters, kernel_size, padding='causal',
                          dilation_rate=dilation_rate, activation='relu')(x)
    conv1 = layers.Dropout(dropout)(conv1)

    # Second dilated conv
    conv2 = layers.Conv1D(filters, kernel_size, padding='causal',
                          dilation_rate=dilation_rate, activation='relu')(conv1)

    # Residual connection (adjust dims if needed)
    if x.shape[-1] != filters:
        x = layers.Conv1D(filters, 1, padding='same')(x)
    out = layers.add([conv2, x])
    out = layers.LayerNormalization()(out)
    return out

def create_tcn_model(input_shape, num_classes=7):
    inputs = Input(shape=input_shape)

    # Stack residual blocks with increasing dilation
    x = tcn_block(inputs, filters=64, dilation_rate=1)
    x = tcn_block(x, filters=64, dilation_rate=2)
    x = tcn_block(x, filters=64, dilation_rate=4)

    # Global pooling to collapse time
    x = layers.GlobalAveragePooling1D()(x)

    # Dense head
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = models.Model(inputs, outputs)
    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model



In [ ]:
tcn_model = create_tcn_model((X_train_no_diff_lstm.shape[1], X_train_no_diff_lstm.shape[2]))

history = tcn_model.fit(X_train_no_diff_lstm, y_train_no_diff_lstm, 
               validation_data=(X_val_no_diff_lstm, y_val_no_diff_lstm), 
               epochs=35,
               batch_size=64,
               verbose=1,
               callbacks=[lr_schedule])

In [ ]:
y_pred_tcn = np.argmax(tcn_model.predict(X_test_no_diff_lstm), axis=1)
print(classification_report(y_test_no_diff_lstm, y_pred_tcn))
cm_tcn = confusion_matrix(y_test_no_diff_lstm, y_pred_tcn)
plt.figure(figsize=(10, 8))
sns.heatmap(cm_tcn, annot=True, fmt='d', cmap='Blues')
plt.title('TCN Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
